# Glass Quality Prediction

A reproducible Extra Trees baseline for the challenge. The workflow deliberately stops with a clear error when the original challenge data is not present; no data or competition score is fabricated.

## Data contract

Place `Train.csv` and `Test.csv` under `data/` at the repository root. `Train.csv` must contain the `class` target; both files must have matching numeric feature columns. The loader validates this contract, missing values, and class cardinality before model fitting.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from glass_quality import (
    TARGET,
    create_features,
    cross_validate_log_loss,
    fit_predict,
    load_data,
    make_model,
    select_feature_columns,
)

In [ ]:
# Missing files or invalid schemas raise an actionable error here.
DATA_DIR = Path("data")
train, test = load_data(DATA_DIR)
train.head(), test.head()

## Row-local feature engineering

All derived values use columns from the same row and never inspect the target. Identifier-like columns, the target, and explicitly redundant or leakage-prone original columns are excluded from model features.

In [ ]:
train = create_features(train)
test = create_features(test)
features = select_feature_columns(train)
print(f"Using {len(features)} features: {features}")

## Exploratory checks

These plots are optional diagnostics and do not alter the model or validation data.

In [ ]:
numeric_features = train[features].select_dtypes(include="number").columns.tolist()
plot_features = numeric_features[:12]
if plot_features:
    axes = train[plot_features].hist(bins=20, figsize=(12, 10))
    plt.tight_layout()
    plt.show()
else:
    print("No numeric features available for plotting.")

In [ ]:
component_features = [
    column for column in features if column.startswith("x_component_")
]
if component_features:
    fig, axes = plt.subplots(
        nrows=1,
        ncols=len(component_features),
        figsize=(4 * len(component_features), 4),
        squeeze=False,
    )
    for axis, column in zip(axes.ravel(), component_features):
        sns.countplot(data=train, x=column, hue=TARGET, ax=axis)
        axis.set_title(column)
    plt.tight_layout()
    plt.show()

## Stratified validation and final fit

Validation uses shuffled `StratifiedKFold` with a fixed seed. Each fold receives a fresh estimator, so validation rows are not used for fitting that fold. Log loss uses the fitted estimator's actual class order.

In [ ]:
model = make_model()
scores = cross_validate_log_loss(model, train, features, target=TARGET, n_splits=5)
print(f"Fold log loss: {scores}")
print(f"Mean log loss: {sum(scores) / len(scores):.6f}")

In [ ]:
predictions = fit_predict(model, train, test, features, target=TARGET)
output_path = Path("extratrees_addv1.xlsx")
predictions.to_excel(output_path, index=False)
print(f"Wrote {len(predictions)} predictions to {output_path}")
predictions.head()

## Limitations

This is a challenge baseline, not a production API. No end-to-end model run is claimed in this repository review because the challenge data is not checked in. Acquire the authorized data separately before running the notebook.